# Campus Twin AI — Model Training (YOLOv8n + LightGBM on BDG2)
Outputs (download these from the Output panel):
  models/best.pt          → person detector (webcam occupancy)
  models/forecast_B*.pkl  → per-building energy forecast
  models/metrics.json     → honest eval metrics
  data/elec_3b.csv        → cleaned 3-building hourly energy (simulator replay)

In [1]:
!pip -q install ultralytics lightgbm pyarrow
import torch, ultralytics
ultralytics.checks()
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

Ultralytics 8.4.161 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 7037.7/8062.4 GB disk)
GPU: True Tesla T4


In [2]:
import glob, os
files = [p for p in glob.glob('/kaggle/input/**/*', recursive=True)
         if p.lower().endswith(('.parquet', '.csv'))]
print("Found files:")
for f in files: print(" ", f)
    

Found files:
  /kaggle/input/datasets/claytonmiller/buildingdatagenomeproject2/water_cleaned.csv
  /kaggle/input/datasets/claytonmiller/buildingdatagenomeproject2/electricity_cleaned.csv
  /kaggle/input/datasets/claytonmiller/buildingdatagenomeproject2/steam_cleaned.csv
  /kaggle/input/datasets/claytonmiller/buildingdatagenomeproject2/water.csv
  /kaggle/input/datasets/claytonmiller/buildingdatagenomeproject2/gas.csv
  /kaggle/input/datasets/claytonmiller/buildingdatagenomeproject2/irrigation_cleaned.csv
  /kaggle/input/datasets/claytonmiller/buildingdatagenomeproject2/weather.csv
  /kaggle/input/datasets/claytonmiller/buildingdatagenomeproject2/hotwater_cleaned.csv
  /kaggle/input/datasets/claytonmiller/buildingdatagenomeproject2/irrigation.csv
  /kaggle/input/datasets/claytonmiller/buildingdatagenomeproject2/gas_cleaned.csv
  /kaggle/input/datasets/claytonmiller/buildingdatagenomeproject2/chilledwater.csv
  /kaggle/input/datasets/claytonmiller/buildingdatagenomeproject2/chilledwater_

In [5]:
import pandas as pd
import os

# Load electricity data
elec_path = "/kaggle/input/datasets/claytonmiller/buildingdatagenomeproject2/electricity_cleaned.csv"

df = pd.read_csv(elec_path)

print("Original shape:", df.shape)

# Convert timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Set timestamp as index
df = df.set_index("timestamp")

print("After setting timestamp:", df.shape)

# Convert all building columns to numeric
df = df.apply(pd.to_numeric, errors="coerce")

# Find the 3 buildings with the most complete observations
best_cols = (
    df.notna()
      .sum()
      .sort_values(ascending=False)
      .head(3)
      .index
      .tolist()
)

print("\nSelected buildings:")
for i, col in enumerate(best_cols, 1):
    print(f"B{i}: {col}")

# Select those buildings
sub = df[best_cols].copy()

# Remove negative values
sub = sub.clip(lower=0)

# Resample to hourly data
sub = sub.resample("h").mean()

# Drop hours where any selected building is missing
sub = sub.dropna(how="any")

# Rename buildings
sub.columns = ["B1", "B2", "B3"]

sub.index.name = "timestamp"

# Create output directory
os.makedirs("/kaggle/working/data", exist_ok=True)

# Save cleaned dataset
output_path = "/kaggle/working/data/elec_3b.csv"
sub.to_csv(output_path)

print("\n" + "="*60)
print("FINAL DATASET")
print("="*60)
print("Shape:", sub.shape)
print("Saved to:", output_path)

print("\nData range:")
print("Start:", sub.index.min())
print("End:", sub.index.max())

print("\nMissing values:")
print(sub.isna().sum())

print("\nEnergy range:")
print("Minimum:", round(sub.min().min(), 3))
print("Maximum:", round(sub.max().max(), 3))

print("\nFirst 5 rows:")
display(sub.head())

Original shape: (17544, 1579)
After setting timestamp: (17544, 1578)

Selected buildings:
B1: Hog_office_Gustavo
B2: Lamb_office_Caitlin
B3: Lamb_health_Ken

FINAL DATASET
Shape: (17544, 3)
Saved to: /kaggle/working/data/elec_3b.csv

Data range:
Start: 2016-01-01 00:00:00
End: 2017-12-31 23:00:00

Missing values:
B1    0
B2    0
B3    0
dtype: int64

Energy range:
Minimum: 0.002
Maximum: 1205.446

First 5 rows:


,B1,B2,B3
timestamp,,,
2016-01-01 00:00:00,52.184,5.8,8.0
2016-01-01 01:00:00,52.184,11.3,14.9
2016-01-01 02:00:00,53.043,11.2,11.2
2016-01-01 03:00:00,52.801,11.9,11.4
2016-01-01 04:00:00,53.061,12.5,11.0


In [6]:
import lightgbm as lgb
import pickle
import json
import os
import time
import numpy as np
from sklearn.metrics import mean_absolute_error

os.makedirs("/kaggle/working/models", exist_ok=True)

FEATS = ['hour', 'dow', 'lag1', 'lag24', 'roll24']

metrics = {}

print("=" * 60)
print("LIGHTGBM TRAINING")
print("=" * 60)

total_start = time.time()

for b in ['B1', 'B2', 'B3']:

    start = time.time()

    print(f"\nTraining {b}...")

    s = sub[b]

    d = pd.DataFrame({'y': s})

    # Time features
    d['hour'] = d.index.hour
    d['dow'] = d.index.dayofweek

    # Lag features
    d['lag1'] = s.shift(1)
    d['lag24'] = s.shift(24)

    # Rolling 24-hour mean
    d['roll24'] = s.shift(1).rolling(24).mean()

    # Remove rows without enough history
    d = d.dropna()

    # Time-based split: 90% train, 10% test
    split = int(len(d) * 0.9)

    X = d[FEATS]
    y = d['y']

    X_train = X.iloc[:split]
    X_test = X.iloc[split:]

    y_train = y.iloc[:split]
    y_test = y.iloc[split:]

    print(f"Train samples: {len(X_train):,}")
    print(f"Test samples : {len(X_test):,}")

    # Model
    mdl = lgb.LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        verbosity=-1
    )

    mdl.fit(X_train, y_train)

    # Prediction
    pred = mdl.predict(X_test)

    # Metrics
    mae = mean_absolute_error(y_test, pred)

    # Avoid division by very small values
    denominator = y_test.clip(lower=1e-3)

    mape = (
        abs(y_test - pred) / denominator
    ).mean() * 100

    residuals = y_test - pred

    resid_std = residuals.std()

    metrics[b] = {
        'mae_kwh': round(float(mae), 3),
        'mape_pct': round(float(mape), 2),
        'resid_std': round(float(resid_std), 3)
    }

    # Save model
    model_path = f"/kaggle/working/models/forecast_{b}.pkl"

    with open(model_path, 'wb') as f:
        pickle.dump(mdl, f)

    elapsed = time.time() - start

    print(f"MAE       : {mae:.3f}")
    print(f"MAPE      : {mape:.2f}%")
    print(f"Residual σ: {resid_std:.3f}")
    print(f"Saved     : {model_path}")
    print(f"Time      : {elapsed:.2f} sec")

# Save metrics
metrics_path = "/kaggle/working/models/metrics.json"

with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

total_time = time.time() - total_start

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

print(f"Total training time: {total_time:.2f} seconds")

print("\nMetrics:")
print(json.dumps(metrics, indent=2))

print("\nSaved files:")
print(" - forecast_B1.pkl")
print(" - forecast_B2.pkl")
print(" - forecast_B3.pkl")
print(" - metrics.json")

LIGHTGBM TRAINING

Training B1...
Train samples: 15,768
Test samples : 1,752
MAE       : 12.091
MAPE      : 1.66%
Residual σ: 23.329
Saved     : /kaggle/working/models/forecast_B1.pkl
Time      : 0.44 sec

Training B2...
Train samples: 15,768
Test samples : 1,752
MAE       : 5.528
MAPE      : 13.80%
Residual σ: 9.215
Saved     : /kaggle/working/models/forecast_B2.pkl
Time      : 0.34 sec

Training B3...
Train samples: 15,768
Test samples : 1,752
MAE       : 1.910
MAPE      : 9.72%
Residual σ: 2.688
Saved     : /kaggle/working/models/forecast_B3.pkl
Time      : 0.34 sec

TRAINING COMPLETE
Total training time: 1.12 seconds

Metrics:
{
  "B1": {
    "mae_kwh": 12.091,
    "mape_pct": 1.66,
    "resid_std": 23.329
  },
  "B2": {
    "mae_kwh": 5.528,
    "mape_pct": 13.8,
    "resid_std": 9.215
  },
  "B3": {
    "mae_kwh": 1.91,
    "mape_pct": 9.72,
    "resid_std": 2.688
  }
}

Saved files:
 - forecast_B1.pkl
 - forecast_B2.pkl
 - forecast_B3.pkl
 - metrics.json


In [7]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

print("YOLO model loaded successfully")

YOLO model loaded successfully


In [8]:
results = model.train(
    data="coco128.yaml",
    classes=[0],
    epochs=40,
    imgsz=640,
    batch=32,
    device=0,
    project="/kaggle/working/yolo",
    name="person"
)

Ultralytics 8.4.161 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=[0], close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco128.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=person, nbs=64, nms=None, opset=None, optimize=False, optimizer=

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/40      6.02G      1.085      2.038       1.19         99        640: 100% ━━━━━━━━━━━━ 4/4 5.2it/s 0.8s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.9it/s 0.7s1.1s
                   all        128        254      0.822      0.669      0.759      0.539

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/40      6.02G      1.133       1.26      1.161        120        640: 100% ━━━━━━━━━━━━ 4/4 5.1it/s 0.8s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 3.2it/s 0.6s0.9s
                   all        128        254       0.83      0.651      0.767      0.546

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/40      6.02G      1.198      1.024      1.157        132        640: 100% ━━━━━━━━━━━━ 4/4 5.2it/s 0.8s0.3s
                 Class     Images  Instances      Bo

In [9]:
#hyparameter tunning 
from ultralytics import YOLO
import pandas as pd, time, os, shutil

experiments = [
    ("baseline", 0.01, "SGD",   640, 32, 40),
    ("low_lr",   0.005, "SGD",  640, 32, 60),
    ("adamw",    0.001, "AdamW",640, 32, 60),
    ("high_res", 0.005, "AdamW",768, 16, 60),
]

results = []
out = "/kaggle/working/yolo_tuning"

for name, lr, opt, imgsz, batch, epochs in experiments:
    print(f"\n========== {name} ==========")
    t = time.time()

    model = YOLO("yolov8n.pt")
    model.train(
        data="coco128.yaml",
        classes=[0],
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        optimizer=opt,
        lr0=lr,
        weight_decay=0.0005,
        device=0,
        seed=42,
        project=out,
        name=name,
        exist_ok=True,
        verbose=False
    )

    path = f"{out}/{name}/weights/best.pt"
    m = YOLO(path).val(
        data="coco128.yaml",
        classes=[0],
        imgsz=imgsz,
        batch=batch,
        device=0,
        verbose=False
    )

    results.append({
        "experiment": name,
        "optimizer": opt,
        "lr": lr,
        "imgsz": imgsz,
        "batch": batch,
        "epochs": epochs,
        "precision": round(float(m.box.mp), 4),
        "recall": round(float(m.box.mr), 4),
        "mAP50": round(float(m.box.map50), 4),
        "mAP50-95": round(float(m.box.map), 4),
        "time_sec": round(time.time()-t, 1),
        "model": path
    })

df = pd.DataFrame(results).sort_values("mAP50-95", ascending=False)

display(df)

# Save results
df.to_csv("/kaggle/working/yolo_tuning_results.csv", index=False)

# Copy best model
best_path = df.iloc[0]["model"]
os.makedirs("/kaggle/working/models", exist_ok=True)
shutil.copy(best_path, "/kaggle/working/models/best_tuned.pt")

print("\nBEST MODEL:")
print(df.iloc[0])
print("\nSaved: /kaggle/working/models/best_tuned.pt")


========== baseline ==========
Ultralytics 8.4.161 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=[0], close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco128.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline, nbs=64, nms=None, opset

,experiment,optimizer,lr,imgsz,batch,epochs,precision,recall,mAP50,mAP50-95,time_sec,model
1,low_lr,SGD,0.005,640,32,60,0.9698,0.8425,0.9197,0.7428,121.1,/kaggle/working/yolo_tuning/low_lr/weights/bes...
2,adamw,AdamW,0.001,640,32,60,0.9238,0.8504,0.9120,0.7266,128.3,/kaggle/working/yolo_tuning/adamw/weights/best.pt
0,baseline,SGD,0.010,640,32,40,0.9124,0.8031,0.8928,0.6906,88.5,/kaggle/working/yolo_tuning/baseline/weights/b...
3,high_res,AdamW,0.005,768,16,60,0.4842,0.3918,0.3593,0.1419,162.0,/kaggle/working/yolo_tuning/high_res/weights/b...



BEST MODEL:
experiment                                               low_lr
optimizer                                                   SGD
lr                                                        0.005
imgsz                                                       640
batch                                                        32
epochs                                                       60
precision                                                0.9698
recall                                                   0.8425
mAP50                                                    0.9197
mAP50-95                                                 0.7428
time_sec                                                  121.1
model         /kaggle/working/yolo_tuning/low_lr/weights/bes...
Name: 1, dtype: object

Saved: /kaggle/working/models/best_tuned.pt


In [10]:
model = YOLO(
    "/kaggle/working/yolo/person/weights/best.pt"
)

metrics_yolo = model.val()

print(metrics_yolo)

Ultralytics 8.4.161 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1656.0±465.1 MB/s, size: 59.4 KB)
val: Scanning /kaggle/working/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 128/128 28.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 3.7it/s 2.2s0.2s
                   all        128        929      0.245      0.104      0.097       0.06
                person         61        254     0.0754      0.953      0.885      0.689
               bicycle          3          6          0          0          0          0
                   car         12         46       0.75     0.0652     0.0488       0.02
            motorcycle          4          5          0          0          0          0
              airplane    

In [11]:
import os
import json
import joblib

print("=== Forecast Models ===")

for b in ["B1", "B2", "B3"]:
    path = f"/kaggle/working/models/forecast_{b}.pkl"
    print(b, "✓" if os.path.exists(path) else "✗", path)

metrics_path = "/kaggle/working/models/metrics.json"

if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)

    print("\n=== Metrics ===")
    print(json.dumps(metrics, indent=2))

=== Forecast Models ===
B1 ✓ /kaggle/working/models/forecast_B1.pkl
B2 ✓ /kaggle/working/models/forecast_B2.pkl
B3 ✓ /kaggle/working/models/forecast_B3.pkl

=== Metrics ===
{
  "B1": {
    "mae_kwh": 12.091,
    "mape_pct": 1.66,
    "resid_std": 23.329
  },
  "B2": {
    "mae_kwh": 5.528,
    "mape_pct": 13.8,
    "resid_std": 9.215
  },
  "B3": {
    "mae_kwh": 1.91,
    "mape_pct": 9.72,
    "resid_std": 2.688
  }
}


In [12]:
import os
import shutil

os.makedirs("/kaggle/working/models", exist_ok=True)

baseline = "/kaggle/working/yolo_tuning/baseline/weights/best.pt"

if os.path.exists(baseline):
    shutil.copy(
        baseline,
        "/kaggle/working/models/best.pt"
    )
    print("Saved:", "/kaggle/working/models/best.pt")
else:
    print("Baseline model not found")

Saved: /kaggle/working/models/best.pt


In [13]:
import shutil
import os

model_dir = "/kaggle/working/models"

zip_path = shutil.make_archive(
    "/kaggle/working/campus_twin_models",
    "zip",
    model_dir
)

print("Created:")
print(zip_path)

Created:
/kaggle/working/campus_twin_models.zip
